# Feature Engineering and Ridge Baseline

This notebook starts from the reproducible interim datasets, defines the chronological evaluation design, and retains only the feature choices supported by walk-forward validation. The 2024/25 test season remains untouched.

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Load prepared data

The expensive raw-table joins live in `src/prem_valuation/data.py`. This notebook loads only the resulting player-season tables.

In [2]:
model_data = pd.read_csv(
    "../data/interim/labelled_player_seasons.csv.gz",
    parse_dates=["season_end_date", "valuation_date", "date_of_birth"],
)

scoring_data = pd.read_csv(
    "../data/interim/scoring_2025_26.csv.gz",
    parse_dates=["season_end_date", "date_of_birth"],
)

In [3]:
def add_engineered_features(data):
    result = data.copy()
    result["age_distance_squared"] = (
        result["age_at_season_end"] - 25
    ) ** 2

    stable_minutes = result["minutes_played"].clip(lower=450)
    result["goals_per_90_stable"] = (
        result["goals"] / stable_minutes * 90
    )
    result["assists_per_90_stable"] = (
        result["assists"] / stable_minutes * 90
    )
    return result

model_data = add_engineered_features(model_data)
scoring_data = add_engineered_features(scoring_data)

## Chronological split

- Train: 2016/17–2022/23
- Validation: 2023/24
- Final untouched test: 2024/25
- Unlabelled scoring population: 2025/26

In [4]:
train_data = model_data.loc[model_data["season"].between(2016, 2022)].copy()
validation_data = model_data.loc[model_data["season"].eq(2023)].copy()
test_data = model_data.loc[model_data["season"].eq(2024)].copy()

for name, dataset in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
    "scoring": scoring_data,
}.items():
    print(name, dataset.shape, sorted(dataset["season"].unique()))

train (3476, 24) [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
validation (534, 24) [np.int64(2023)]
test (477, 24) [np.int64(2024)]
scoring (537, 21) [np.int64(2025)]


## Feature specification

Identifiers, names, dates, and valuation timing metadata are excluded. Citizenship is excluded because it is high-cardinality and reflects a current rather than historical profile. `season` is retained only for splitting, not as a feature.

In [5]:
target = "market_value_in_eur"

original_numeric_features = [
    "appearances", "minutes_played", "goals", "assists",
    "yellow_cards", "red_cards", "clubs_played_for",
    "height_in_cm", "age_at_season_end",
]
original_categorical_features = ["position", "foot"]

selected_numeric_features = original_numeric_features + [
    "age_distance_squared"
]
selected_categorical_features = ["position", "sub_position", "foot"]

rate_numeric_features = selected_numeric_features + [
    "goals_per_90_stable", "assists_per_90_stable"
]
selected_features = selected_numeric_features + selected_categorical_features

In [6]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

## Reusable temporal evaluation

Each fold trains only on earlier seasons and validates on the next season. MAE is the primary metric; RMSE and R² remain diagnostic.

In [7]:
def make_ridge_model(numerical_columns, categorical_columns, alpha=1.0):
    preprocessor = ColumnTransformer(transformers=[
        ("numeric", numeric_transformer, numerical_columns),
        ("categorical", categorical_transformer, categorical_columns),
    ])
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=alpha)),
    ])


def evaluate_predictions(actual, predicted):
    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": root_mean_squared_error(actual, predicted),
        "r2": r2_score(actual, predicted),
    }


def walk_forward_evaluate(numerical_columns, categorical_columns, alpha=1.0):
    rows = []
    feature_columns = numerical_columns + categorical_columns

    for validation_season in range(2019, 2024):
        fold_train = model_data.loc[
            model_data["season"].between(2016, validation_season - 1)
        ]
        fold_validation = model_data.loc[
            model_data["season"].eq(validation_season)
        ]
        model = make_ridge_model(
            numerical_columns, categorical_columns, alpha=alpha
        )
        model.fit(fold_train[feature_columns], fold_train[target])
        predictions = np.clip(
            model.predict(fold_validation[feature_columns]), 0, None
        )
        rows.append({
            "validation_season": validation_season,
            **evaluate_predictions(fold_validation[target], predictions),
        })

    return pd.DataFrame(rows)

## Current selected Ridge model

In [8]:
selected_model = make_ridge_model(
    selected_numeric_features, selected_categorical_features
)
selected_model.fit(train_data[selected_features], train_data[target])
validation_predictions = np.clip(
    selected_model.predict(validation_data[selected_features]), 0, None
)

pd.Series(
    evaluate_predictions(validation_data[target], validation_predictions),
    name="2023/24 validation",
)

mae     1.037542e+07
rmse    1.584161e+07
r2      5.012966e-01
Name: 2023/24 validation, dtype: float64

In [9]:
original_cv = walk_forward_evaluate(
    original_numeric_features, original_categorical_features
)
age_cv = walk_forward_evaluate(
    selected_numeric_features, original_categorical_features
)
selected_cv = walk_forward_evaluate(
    selected_numeric_features, selected_categorical_features
)
rates_cv = walk_forward_evaluate(
    rate_numeric_features, selected_categorical_features
)

cv_summary = pd.DataFrame({
    "original": original_cv[["mae", "rmse", "r2"]].mean(),
    "age_curve": age_cv[["mae", "rmse", "r2"]].mean(),
    "age_and_subposition": selected_cv[["mae", "rmse", "r2"]].mean(),
    "plus_stable_rates": rates_cv[["mae", "rmse", "r2"]].mean(),
}).T

cv_summary

,mae,rmse,r2
original,9.625449e+06,1.439202e+07,0.477009
age_curve,9.412778e+06,1.420875e+07,0.490398
age_and_subposition,9.306918e+06,1.404231e+07,0.502172
plus_stable_rates,9.292803e+06,1.402092e+07,0.503767


## Experiment log

| Experiment | 2023/24 result | Decision | Reason |
|---|---:|---|---|
| Median dummy | MAE €15.45m | Benchmark | Minimum useful baseline. |
| Log-target Ridge | MAE €11.69m | Rejected | Worse metrics and implausible €288m Palmer prediction. |
| Raw-target Ridge | MAE €10.89m | Retained foundation | Beat dummy and log target. |
| Numeric season trend | MAE €11.44m | Rejected | Improved RMSE/R² but worsened primary MAE. |
| Squared distance from age 25 | MAE €10.69m | Retained | Captures the observed nonlinear age-value curve. |
| Sub-position | MAE €10.38m | Retained | Improved all metrics and remained useful across temporal folds. |
| Stabilised goals/assists per 90 | MAE €10.35m | Rejected | Only marginal validation gain; mixed results over five folds. |

Walk-forward averages improved from MAE €9.63m / R² 0.477 for the original features to MAE €9.31m / R² 0.502 after adding the age curve and sub-position.

## Ranking eligibility and next step

The model trains on all player-seasons, but final bargain rankings will separate established players (900+ minutes), emerging prospects (450–899), and insufficient samples (under 450).

Next: tune Ridge `alpha` with walk-forward validation. The 2024/25 test set must remain untouched until model and feature decisions are frozen.